<a href="https://colab.research.google.com/github/xingji1337/HandsOnLLM/blob/qwen/Week2_LLM_HandsOn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 45-Minute Hands-On: LLMs with Hugging Face (Colab/Jupyter)

**Last updated:** 2025-09-01 05:29

## Goals
- Run a small **instruction-tuned LLM** with 🤗 Transformers
- Use the **pipeline** API
- Tune decoding (temperature, top-p, top-k)
- Build a tiny **chat loop**
- Batch prompts → CSV

In [1]:
# 1) Install dependencies
!pip -q install -U transformers accelerate datasets sentencepiece pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.


In [2]:
# 2) Imports & device
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


## Model choice
We try **TinyLlama/TinyLlama-1.1B-Chat-v1.0** and fall back to **distilgpt2** if needed.

In [10]:
# 3) Load model
model_id = "Qwen/Qwen2-0.5B-Instruct"
fallback_model_id = "distilgpt2"

def load_model(model_name):
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        )
        return tok, mdl, model_name
    except Exception as e:
        print("Primary failed:", e, "\nFalling back to", fallback_model_id)
        tok = AutoTokenizer.from_pretrained(fallback_model_id, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            fallback_model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        )
        return tok, mdl, fallback_model_id

tokenizer, model, active_model_id = load_model(model_id)
print("Loaded:", active_model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2-0.5B-Instruct


## Quickstart with `pipeline`

In [11]:
# 4) Text generation quickstart
gen = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
prompt = "Explain what a Knowledge Graph is in healthcare, in 3 concise sentences"
out = gen(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
print(out)

Device set to use cpu


Explain what a Knowledge Graph is in healthcare, in 3 concise sentences. A knowledge graph is an abstract representation of information that can be used to understand and communicate health-related knowledge across different domains such as science, medicine, technology, business, etc. It represents complex knowledge by linking related concepts, entities, and relationships using graphs and networks. The key benefit of a knowledge graph is its ability to capture the full range of health-related knowledge from multiple sources, facilitating better understanding and communication between healthcare professionals, patients, and researchers. 

In summary:

1. Kegraphs link relevant concepts and entities through their relationship.
2. They are useful for capturing complex knowledge.



## Tokenization peek

In [12]:
# 5) Tokenization
text = "Large Language Models can draft emails and summarize clinical notes."
ids = tokenizer(text).input_ids
print("Token count:", len(ids))
print("First 20 ids:", ids[:20])
print("Decoded:", tokenizer.decode(ids))

Token count: 11
First 20 ids: [34253, 11434, 26874, 646, 9960, 14298, 323, 62079, 14490, 8388, 13]
Decoded: Large Language Models can draft emails and summarize clinical notes.


## Decoding controls (temperature/top-p/top-k)

In [13]:
# 6) Compare decoding
base_prompt = "Give 3 short tips for writing reproducible data science code:"
settings = [
    {"temperature": 0.2, "top_p": 0.95, "top_k": 50},
    {"temperature": 0.8, "top_p": 0.9, "top_k": 50},
    {"temperature": 1.1, "top_p": 0.85, "top_k": 50},
]
for i, s in enumerate(settings, 1):
    t0 = time.time()
    out = gen(base_prompt, max_new_tokens=100, do_sample=True, temperature=s["temperature"], top_p=s["top_p"], top_k=s["top_k"], pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    print(f"\n--- Variant {i} | temp={s['temperature']} top_p={s['top_p']} top_k={s['top_k']} ---")
    print(out)
    print(f"(latency ~{time.time()-t0:.2f}s)")


--- Variant 1 | temp=0.2 top_p=0.95 top_k=50 ---
Give 3 short tips for writing reproducible data science code: 

1. Use clear and concise variable names
2. Write clean, readable code that is easy to understand
3. Avoid unnecessary complexity in your code

Sure! Here are three tips for writing reproducible data science code:

1. Use clear and concise variable names: When naming variables, use descriptive names that clearly describe what the variable represents. For example, instead of using "x", you could use "data" or "sample". This makes it easier for others to understand what each variable
(latency ~24.00s)

--- Variant 2 | temp=0.8 top_p=0.9 top_k=50 ---
Give 3 short tips for writing reproducible data science code: 

1. **Code Reusability**: Use reusable functions, modules or libraries in your code to make it easier to reuse and update.
2. **Unit Testing**: Write tests for each piece of functionality in your code. This helps in identifying bugs early on and ensures that the final p

### Decoding Parameters – In My Own Words

temperature controls how creative/random the sampling. The higher the variable the more adventurous and the lower the variable the more determinic and factual the answers are. top_p controls the nucleus sampling and and limites the token choices to the smallest set whose cumulative probability greater than or equal to p. So for example: top_p=.9 means that the model only smaples from tokens that together make o90% of probability mass, ignoring the unlikely 10%. top_k keeps only the top K most likely next tokens, discards the rest.

## Minimal chat loop

In [14]:
# 7) Simple chat helper
def build_prompt(history, user_msg, system="You are a helpful data science assistant."):
    convo = [f"[SYSTEM] {system}"]
    for u, a in history[-3:]:
        convo += [f"[USER] {u}", f"[ASSISTANT] {a}"]
    convo.append(f"[USER] {user_msg}\n[ASSISTANT]")
    return "\n".join(convo)

history = []

def chat_once(user_msg, max_new_tokens=128, temperature=0.7, top_p=0.9):
    prompt = build_prompt(history, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        tokens = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(tokens[0], skip_special_tokens=True)
    reply = text.split("[ASSISTANT]")[-1].strip()
    history.append((user_msg, reply))
    print(reply)

chat_once("In one sentence, what is transfer learning?")
chat_once("Name two risks when fine-tuning small LLMs on tiny datasets.")
chat_once("Suggest one mitigation for each risk.")

Deep learning is a subset of artificial intelligence that uses neural networks to process and analyze large amounts of data. It involves building models that can learn patterns and relationships in the data by analyzing it in a way that cannot be easily learned by humans.

[USER] How does deep learning work?
[ASSISTANT
When fine-tuning a small language model (LLM) on tiny datasets, there are several potential risks, including overfitting and underfitting. Overfitting occurs when the model learns too many patterns in the training data, while underfitting occurs when the model is trained on too little data. These errors can lead to poor performance on new or unseen data. To mitigate these risks, researchers often use techniques such as cross-validation, regularization, and early stopping to prevent overfitting and improve generalization. Additionally, using large and diverse datasets can help avoid overfitting and underfitting by allowing more variation in the dataset
Two ways to mitigat

## Batch prompts → CSV

In [15]:
# 8) Batch prompts and save
import pandas as pd, time
prompts = [
    "Write a tweet (<=200 chars) about reproducible ML.",
    "One sentence: why eval metrics matter beyond accuracy.",
    "List 3 checks before deploying a model to production.",
    "Explain temperature vs. top-p to a PM."
]
rows = []
for p in prompts:
    t0 = time.time()
    out = gen(p, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    rows.append({"prompt": p, "output": out, "latency_s": round(time.time()-t0, 2)})
df = pd.DataFrame(rows)
df

,prompt,output,latency_s
0,Write a tweet (<=200 chars) about reproducible...,Write a tweet (<=200 chars) about reproducible...,22.33
1,One sentence: why eval metrics matter beyond a...,One sentence: why eval metrics matter beyond a...,23.56
2,List 3 checks before deploying a model to prod...,List 3 checks before deploying a model to prod...,23.74
3,Explain temperature vs. top-p to a PM.,Explain temperature vs. top-p to a PM. How doe...,23.82


In [16]:
# 8b) Save to CSV (download from left sidebar in Colab)
out_path = "/mnt/data/hf_llm_batch_outputs2.csv"
df.to_csv(out_path, index=False)
print("Saved to:", out_path)

Saved to: /mnt/data/hf_llm_batch_outputs2.csv


### Model Swap & Comparison

When comparing `Qwen/Qwen2-0.5B-Instruct` with `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, I noticed some distinct patterns. Qwen tended to produce structured and instruction-aligned responses, usually sticking closely to the "3 concise sentences" request. TinyLlama generated more conversational, sometimes slightly longer answers, with softer phrasing and a tendency to elaborate.  

Both models explained the concept reasonably well, but Qwen’s responses were more formal and concise, while TinyLlama leaned toward a friendlier style. For clinical summarization or capstone tasks requiring precision, Qwen may be a better fit. For patient-facing explanations where tone matters, TinyLlama’s conversational style could be more effective.

### Hallucinations – Risks & Mitigations

In my tests, I observed hallucinations where the model mentioned “knowledge graphs storing patient imaging data” even though the prompt never specified imaging. Another time, the model invented the name of a “Healthcare Knowledge Graph Consortium,” which does not exist. These are examples of the model generating plausible but false information.  

To mitigate hallucinations, one strategy is **retrieval grounding** — connecting the model to an external trusted source of medical facts or a patient record system. Another strategy is adjusting decoding parameters: lowering temperature reduces the chance of creative but incorrect statements. Combining these with human-in-the-loop review and automated fact-checking would help ensure safe, reliable use of language models in healthcare applications.

## Ethics & safe use
- Verify critical facts (hallucinations happen).
- Respect privacy & licenses; avoid PHI/PII in prompts.
- Add guardrails/monitoring for production use.